# Held-out spatial-minus-temporal top-\(m\) heatmaps

This notebook compares **participant-held-out top-\(m\)** decoding for spatial and
temporal MS-AA at exactly matched normalized component ratios.

Important features of the revised analysis:

- archetypes are selected on one participant half and evaluated on the other;
- spatial and temporal top-\(m\) subsets are selected independently within their
  respective decompositions;
- A→B and B→A directions are averaged within each outer split before summarization;
- spatial and temporal models are compared only at exact normalized component ratios,
  \(K_{\mathrm{spatial}}/700 = K_{\mathrm{temporal}}/300\);
- saturated cells with \(m>K\) are masked;
- no interpolation is used;
- asterisks mark cells with FDR-corrected \(q<0.05\) from two-sided permutation tests
  comparing the distributions of spatial and temporal outer-split accuracy estimates.

Because outer cross-validation splits reuse participants, the permutation tests should
be interpreted as evidence that a spatial-temporal difference is stable across
cross-validation resamples, rather than as independent-subject population inference.


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SPATIAL_RUN_CSV = Path(
    "msaa_heldout_topm_outputs_spatial_across_fullgrid/"
    "heldout_topm_crossfit_runs.csv"
)
TEMPORAL_RUN_CSV = Path(
    "msaa_heldout_topm_outputs_temporal_across_fullgrid/"
    "heldout_topm_crossfit_runs.csv"
)

OUTPUT_DIR = Path("outputs/paper_supplement")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_PNG = OUTPUT_DIR / "Supplement_heldout_spatial_minus_temporal_heatmaps.png"
FIG_PDF = OUTPUT_DIR / "Supplement_heldout_spatial_minus_temporal_heatmaps.pdf"
SOURCE_CSV = OUTPUT_DIR / "Supplement_heldout_spatial_minus_temporal_heatmaps_source.csv"

CONDITIONS = ["intact", "word", "rest"]
CONDITION_TITLES = {
    "intact": "Intact",
    "word": "Word-scrambled",
    "rest": "Rest",
}
TOP_M_VALUES = [1, 2, 3, 5, 7, 10, 15, 20, 25, 30, 40, 50, 75, 100]

DIMENSIONS = {"spatial": 700, "temporal": 300}


# Statistical display settings.
N_PERMUTATIONS = 5000
FDR_ALPHA = 0.05
PERMUTATION_SEED = 42

# Show raw spatial/temporal K values beneath a few anchor ratios on the x-axis.
COMPONENT_LABEL_RATIOS = [0.01, 0.09, 0.18, 0.30, 1.00]



In [ ]:

if not SPATIAL_RUN_CSV.exists():
    raise FileNotFoundError(
        f"Missing {SPATIAL_RUN_CSV}\n"
        "Run 004b_expand_spatial_heldout_topm_fullgrid.ipynb first."
    )
if not TEMPORAL_RUN_CSV.exists():
    raise FileNotFoundError(
        f"Missing {TEMPORAL_RUN_CSV}\n"
        "Run 004_temporal_heldout_topm_fullgrid.ipynb first."
    )

spatial_runs = pd.read_csv(SPATIAL_RUN_CSV)
temporal_runs = pd.read_csv(TEMPORAL_RUN_CSV)

print("Spatial rows:", len(spatial_runs))
print("Temporal rows:", len(temporal_runs))
print("Spatial top-m:", sorted(spatial_runs["top_m"].astype(int).unique()))
print("Temporal top-m:", sorted(temporal_runs["top_m"].astype(int).unique()))


In [ ]:

def sem(x):
    x = pd.Series(x).dropna()
    return np.nan if len(x) <= 1 else float(x.std(ddof=1) / np.sqrt(len(x)))

def summarize_outer_paired(run_df, analysis):
    df = run_df.copy()
    df = df[df["analysis_type"].astype(str) == analysis].copy()

    # Average A->B and B->A within each outer participant split first.
    outer = (
        df.groupby(
            ["condition", "K", "top_m", "outer_rep"],
            as_index=False,
        )
        .agg(
            accuracy=("mean", "mean"),
            full_accuracy=("full_mean_same_eval", "mean"),
        )
    )

    return (
        outer.groupby(["condition", "K", "top_m"], as_index=False)
        .agg(
            accuracy=("accuracy", "mean"),
            sem=("accuracy", sem),
            n_outer=("accuracy", "size"),
            full_accuracy=("full_accuracy", "mean"),
        )
    )

spatial = summarize_outer_paired(spatial_runs, "spatial")
temporal = summarize_outer_paired(temporal_runs, "temporal")

display(spatial.head())
display(temporal.head())


In [ ]:
# ============================================================
# Outer-split distributions and permutation-test helpers
# ============================================================

def outer_split_accuracy(run_df, analysis):
    """
    Average A->B and B->A within each outer split, but retain the 20
    outer-split estimates for significance testing.
    """
    df = run_df[
        run_df["analysis_type"].astype(str) == analysis
    ].copy()

    return (
        df.groupby(
            ["condition", "K", "top_m", "outer_rep"],
            as_index=False,
        )
        .agg(accuracy=("mean", "mean"))
    )


spatial_outer = outer_split_accuracy(spatial_runs, "spatial")
temporal_outer = outer_split_accuracy(temporal_runs, "temporal")


def permutation_pvalue(x, y, n_perm=5000, seed=42):
    """
    Two-sided independent-label permutation test on outer-split accuracy
    estimates. Spatial and temporal outer splits are not paired because the
    two analyses were generated with different deterministic split schemes.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    if len(x) < 2 or len(y) < 2:
        return np.nan

    observed = abs(x.mean() - y.mean())
    pooled = np.concatenate([x, y])
    nx = len(x)

    rng = np.random.default_rng(int(seed))
    extreme = 0

    for _ in range(int(n_perm)):
        perm = rng.permutation(pooled)
        diff = abs(
            perm[:nx].mean()
            - perm[nx:].mean()
        )
        if diff >= observed:
            extreme += 1

    return (extreme + 1) / (int(n_perm) + 1)


def bh_fdr(pvalues):
    """
    Benjamini-Hochberg FDR correction across the displayed heatmap cells.
    """
    p = np.asarray(pvalues, dtype=float)
    q = np.full(len(p), np.nan, dtype=float)

    valid = np.where(np.isfinite(p))[0]
    if len(valid) == 0:
        return q

    pv = p[valid]
    order = np.argsort(pv)
    ranked = pv[order]

    m = len(ranked)
    adjusted = ranked * m / np.arange(1, m + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.minimum(adjusted, 1.0)

    q_valid = np.empty(m, dtype=float)
    q_valid[order] = adjusted
    q[valid] = q_valid

    return q


In [ ]:

# Exact spatial/temporal normalized component-ratio matches.
# Every row satisfies spatial_K / 700 == temporal_K / 300 exactly.
EXACT_PAIRS = [
    (7,   3),    # 0.01
    (14,  6),    # 0.02
    (21,  9),    # 0.03
    (35, 15),    # 0.05
    (42, 18),    # 0.06
    (63, 27),    # 0.09
    (70, 30),    # 0.10
    (105, 45),   # 0.15
    (126, 54),   # 0.18
    (140, 60),   # 0.20
    (175, 75),   # 0.25
    (189, 81),   # 0.27
    (210, 90),   # 0.30
    (252, 108),  # 0.36
    (700, 300),  # 1.00
]

pair_table = pd.DataFrame(EXACT_PAIRS, columns=["spatial_K", "temporal_K"])
pair_table["spatial_ratio"] = (
    pair_table["spatial_K"] / DIMENSIONS["spatial"]
)
pair_table["temporal_ratio"] = (
    pair_table["temporal_K"] / DIMENSIONS["temporal"]
)
pair_table["matched_ratio"] = pair_table["spatial_ratio"]
pair_table["ratio_difference"] = np.abs(
    pair_table["spatial_ratio"] - pair_table["temporal_ratio"]
)

# These should be mathematically identical; fail loudly if the table is edited
# incorrectly in the future.
if not np.allclose(
    pair_table["spatial_ratio"],
    pair_table["temporal_ratio"],
    rtol=0.0,
    atol=1e-12,
):
    raise ValueError("EXACT_PAIRS contains a non-exact spatial/temporal ratio match.")

# Report any model orders that are not yet present in the held-out result files.
spatial_available = set(spatial["K"].astype(int))
temporal_available = set(temporal["K"].astype(int))
missing_spatial = sorted(set(pair_table["spatial_K"]) - spatial_available)
missing_temporal = sorted(set(pair_table["temporal_K"]) - temporal_available)

if missing_spatial:
    print("WARNING: missing spatial held-out K values:", missing_spatial)
if missing_temporal:
    print("WARNING: missing temporal held-out K values:", missing_temporal)

display(pair_table)


In [ ]:

def build_difference_table(spatial, temporal, pair_table):
    rows = []

    for p in pair_table.itertuples(index=False):
        sK = int(p.spatial_K)
        tK = int(p.temporal_K)

        for condition in CONDITIONS:
            for top_m in TOP_M_VALUES:
                # Match the submitted heatmap's saturation mask.
                if top_m > sK or top_m > tK:
                    continue

                s = spatial[
                    (spatial["condition"].astype(str) == condition)
                    & (spatial["K"].astype(int) == sK)
                    & (spatial["top_m"].astype(int) == top_m)
                ]
                t = temporal[
                    (temporal["condition"].astype(str) == condition)
                    & (temporal["K"].astype(int) == tK)
                    & (temporal["top_m"].astype(int) == top_m)
                ]

                if len(s) == 0 or len(t) == 0:
                    continue

                s_acc = float(s["accuracy"].iloc[0])
                t_acc = float(t["accuracy"].iloc[0])

                rows.append({
                    "condition": condition,
                    "top_m": int(top_m),
                    "spatial_K": sK,
                    "temporal_K": tK,
                    "spatial_ratio": float(p.spatial_ratio),
                    "temporal_ratio": float(p.temporal_ratio),
                    "matched_ratio": float(p.matched_ratio),
                    "ratio_difference": float(p.ratio_difference),
                    "spatial_heldout_accuracy": s_acc,
                    "temporal_heldout_accuracy": t_acc,
                    "difference_spatial_minus_temporal": s_acc - t_acc,
                    "spatial_sem": float(s["sem"].iloc[0]),
                    "temporal_sem": float(t["sem"].iloc[0]),
                    "spatial_n_outer": int(s["n_outer"].iloc[0]),
                    "temporal_n_outer": int(t["n_outer"].iloc[0]),
                })

    return pd.DataFrame(rows)

diff_df = build_difference_table(spatial, temporal, pair_table)
print("Difference rows:", len(diff_df))

coverage = (
    diff_df.groupby("condition")
    .agg(
        n_cells=("difference_spatial_minus_temporal", "size"),
        n_pairs=("spatial_K", "nunique"),
        n_top_m=("top_m", "nunique"),
    )
)
display(coverage)


In [ ]:
# ============================================================
# Spatial-vs-temporal permutation tests + FDR correction
# ============================================================

pvals = []

for row in diff_df.itertuples(index=False):
    s = spatial_outer[
        (spatial_outer["condition"].astype(str) == str(row.condition))
        & (spatial_outer["K"].astype(int) == int(row.spatial_K))
        & (spatial_outer["top_m"].astype(int) == int(row.top_m))
    ]["accuracy"].to_numpy()

    t = temporal_outer[
        (temporal_outer["condition"].astype(str) == str(row.condition))
        & (temporal_outer["K"].astype(int) == int(row.temporal_K))
        & (temporal_outer["top_m"].astype(int) == int(row.top_m))
    ]["accuracy"].to_numpy()

    # Require complete 20-split coverage on both sides before testing.
    if len(s) != 20 or len(t) != 20:
        pvals.append(np.nan)
        continue

    seed = (
        PERMUTATION_SEED
        + 100000 * CONDITIONS.index(str(row.condition))
        + 1000 * int(row.spatial_K)
        + int(row.top_m)
    )

    pvals.append(
        permutation_pvalue(
            s,
            t,
            n_perm=N_PERMUTATIONS,
            seed=seed,
        )
    )

diff_df["p_perm"] = pvals
diff_df["q_fdr"] = bh_fdr(diff_df["p_perm"].values)
diff_df["significant_fdr05"] = diff_df["q_fdr"] < FDR_ALPHA

# Require both statistical support and a practically meaningful effect.
# 0.01 = one percentage point of held-out decoding accuracy.
MIN_EFFECT = 0.005

diff_df["robust_effect"] = (
    diff_df["significant_fdr05"]
    & (
        diff_df["difference_spatial_minus_temporal"].abs()
        >= MIN_EFFECT
    )
)

diff_df.to_csv(SOURCE_CSV, index=False)

print(
    "FDR-significant cells:",
    int(diff_df["significant_fdr05"].sum()),
    "/",
    int(diff_df["q_fdr"].notna().sum()),
)
print("Saved:", SOURCE_CSV)

display(
    diff_df.sort_values(
        ["q_fdr", "condition", "matched_ratio", "top_m"]
    ).head(30)
)


In [ ]:

def make_heat(condition):
    heat = np.full((len(TOP_M_VALUES), len(pair_table)), np.nan)

    sub = diff_df[diff_df["condition"] == condition]

    for i, m in enumerate(TOP_M_VALUES):
        for j, p in enumerate(pair_table.itertuples(index=False)):
            vals = sub[
                (sub["top_m"].astype(int) == int(m))
                & (sub["spatial_K"].astype(int) == int(p.spatial_K))
                & (sub["temporal_K"].astype(int) == int(p.temporal_K))
            ]["difference_spatial_minus_temporal"]
            if len(vals):
                heat[i, j] = float(vals.iloc[0])

    return heat

heats = {condition: make_heat(condition) for condition in CONDITIONS}
finite = np.concatenate([
    h[np.isfinite(h)] for h in heats.values() if np.isfinite(h).any()
])
if len(finite) == 0:
    raise RuntimeError("No matched held-out cells were available to plot.")

max_abs = float(np.max(np.abs(finite)))
print("Shared color range:", -max_abs, max_abs)


In [ ]:
def make_robust_heat(condition):
    robust = np.zeros(
        (len(TOP_M_VALUES), len(pair_table)),
        dtype=bool,
    )

    sub = diff_df[
        diff_df["condition"].astype(str) == str(condition)
    ]

    for i, m in enumerate(TOP_M_VALUES):
        for j, p in enumerate(pair_table.itertuples(index=False)):
            vals = sub[
                (sub["top_m"].astype(int) == int(m))
                & (sub["spatial_K"].astype(int) == int(p.spatial_K))
                & (sub["temporal_K"].astype(int) == int(p.temporal_K))
            ]

            if len(vals):
                robust[i, j] = bool(
                    vals["robust_effect"].iloc[0]
                )

    return robust


robust_heats = {
    condition: make_robust_heat(condition)
    for condition in CONDITIONS
}

In [ ]:
fig, axes = plt.subplots(
    3, 1,
    figsize=(13, 9),
    sharex=True,
    constrained_layout=False,
)

ims = []

for ax, condition in zip(axes, CONDITIONS):
    heat = heats[condition]
    robust = robust_heats[condition]

    im = ax.imshow(
        heat,
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        cmap="RdBu_r",
        vmin=-max_abs,
        vmax=max_abs,
    )
    ims.append(im)

    ax.set_title(
        CONDITION_TITLES[condition],
        fontsize=15,
        fontweight="bold",
    )
    ax.set_ylabel("top-m")
    ax.set_yticks(np.arange(len(TOP_M_VALUES)))
    ax.set_yticklabels(TOP_M_VALUES)

    # Open circles mark differences that are both FDR-supported
    # and at least 0.01 in absolute held-out accuracy.
    for i in range(robust.shape[0]):
        for j in range(robust.shape[1]):
            if not robust[i, j]:
                continue

            val = heat[i, j]

            ax.plot(
                j,
                i,
                marker="o",
                markersize=4.5,
                markerfacecolor="none",
                markeredgecolor=(
                    "white"
                    if np.isfinite(val) and abs(val) > 0.40 * max_abs
                    else "black"
                ),
                markeredgewidth=1.1,
                linestyle="None",
            )

# Ratio labels for all columns; raw K values only at selected anchors.
labels = []

for p in pair_table.itertuples(index=False):
    ratio = float(p.matched_ratio)
    label = f"{ratio:.2f}"

    if any(
        np.isclose(ratio, anchor, atol=1e-12)
        for anchor in COMPONENT_LABEL_RATIOS
    ):
        label += (
            f"\nS{int(p.spatial_K)}/T{int(p.temporal_K)}"
        )

    labels.append(label)

axes[-1].set_xticks(np.arange(len(pair_table)))
axes[-1].set_xticklabels(
    labels,
    rotation=45,
    ha="right",
    fontsize=9,
)
axes[-1].set_xlabel(
    "Matched component ratio (K / D)\n"
    "Selected labels show spatial K / temporal K"
)

fig.suptitle(
    "Held-out spatial minus temporal top-m timepoint identification",
    fontsize=17,
    fontweight="bold",
    y=0.985,
)

fig.subplots_adjust(
    left=0.10,
    right=0.88,
    top=0.92,
    bottom=0.15,
    hspace=0.18,
)

cax = fig.add_axes([0.90, 0.20, 0.018, 0.60])
cb = fig.colorbar(ims[0], cax=cax)
cb.set_label("Spatial - temporal held-out accuracy")

fig.text(
    0.88,
    0.115,
    r"$\circ$ FDR $q<0.05$ and $|\Delta accuracy|\geq0.005$",
    ha="right",
    va="bottom",
    fontsize=9,
)

fig.savefig(FIG_PNG, dpi=300, bbox_inches="tight")
fig.savefig(FIG_PDF, bbox_inches="tight")
plt.show()

print("Saved:", FIG_PNG)
print("Saved:", FIG_PDF)


## Interpretation

A positive cell means that the participant-held-out spatial top-\(m\) reconstruction
had higher mean timepoint-identification accuracy than the participant-held-out
temporal top-\(m\) reconstruction at the same normalized component ratio. Negative
cells favor temporal MS-AA.

Asterisks denote cells with Benjamini-Hochberg FDR-corrected \(q<0.05\) from
two-sided permutation tests comparing the distributions of outer-split spatial and
temporal accuracy estimates. Spatial and temporal outer splits are treated as
independent resamples because the two analyses used different deterministic split
schemes.

Because outer cross-validation splits reuse participants, these tests quantify the
stability of the spatial-temporal difference across cross-validation resamples rather
than independent-subject population-level inference.

Selected x-axis labels show the corresponding raw spatial and temporal model orders
(\(S_K/T_K\)) for anchor component ratios.


## Sparse-regime and high-\(m\) saturation controls

The equal-\(m\) heatmap compares the same absolute number of archetypes across
spatial and temporal MS-AA. Because exactly matched model orders satisfy
\(K_{\mathrm{spatial}}=(7/3)K_{\mathrm{temporal}}\), the same \(m\) always
constitutes a larger fraction of the temporal decomposition. At high \(m\),
the temporal reconstruction can therefore approach its full model while the
spatial reconstruction remains substantially sparser.

The cells below provide two complementary views without replacing the full
equal-component-budget analysis:

1. a **sparse-regime figure** restricted to \(m\leq15\);
2. a **saturation-masked diagnostic** that suppresses cells for which
   \(m/K_{\mathrm{temporal}}>0.50\).

The second figure is a diagnostic, not a fully matched-fraction comparison.
A table of exact matched-fraction subset sizes is also exported to identify
the additional top-\(m\) evaluations required for that analysis.

In [ ]:
# HIGH_M_CONTROL_PATCH_V1

SPARSE_TOP_M_VALUES = [1, 2, 3, 5, 7, 10, 15]
MAX_TEMPORAL_SUBSET_FRACTION = 0.50
MATCHED_FRACTION_TARGETS = [0.10, 0.25, 0.50]

FIG_SPARSE_PNG = (
    OUTPUT_DIR
    / "Supplement_heldout_spatial_minus_temporal_sparse_m15.png"
)
FIG_SPARSE_PDF = (
    OUTPUT_DIR
    / "Supplement_heldout_spatial_minus_temporal_sparse_m15.pdf"
)

FIG_MASKED_PNG = (
    OUTPUT_DIR
    / "Supplement_heldout_spatial_minus_temporal_temporal_fraction_le50.png"
)
FIG_MASKED_PDF = (
    OUTPUT_DIR
    / "Supplement_heldout_spatial_minus_temporal_temporal_fraction_le50.pdf"
)

MATCHED_FRACTION_REQUIREMENTS_CSV = (
    OUTPUT_DIR
    / "Supplement_matched_fraction_topm_requirements.csv"
)


def make_control_arrays(
    condition,
    top_m_values,
    max_temporal_fraction=None,
):
    effect = np.full(
        (len(top_m_values), len(pair_table)),
        np.nan,
        dtype=float,
    )
    robust = np.zeros(
        (len(top_m_values), len(pair_table)),
        dtype=bool,
    )

    sub = diff_df[
        diff_df["condition"].astype(str) == str(condition)
    ]

    for i, m in enumerate(top_m_values):
        for j, p in enumerate(pair_table.itertuples(index=False)):
            spatial_K = int(p.spatial_K)
            temporal_K = int(p.temporal_K)

            if m > spatial_K or m > temporal_K:
                continue

            temporal_fraction = m / temporal_K

            if (
                max_temporal_fraction is not None
                and temporal_fraction > max_temporal_fraction
            ):
                continue

            vals = sub[
                (sub["top_m"].astype(int) == int(m))
                & (sub["spatial_K"].astype(int) == spatial_K)
                & (sub["temporal_K"].astype(int) == temporal_K)
            ]

            if len(vals) == 0:
                continue

            effect[i, j] = float(
                vals["difference_spatial_minus_temporal"].iloc[0]
            )

            if "robust_effect" in vals.columns:
                robust[i, j] = bool(
                    vals["robust_effect"].iloc[0]
                )

    return effect, robust


def component_ratio_labels():
    labels = []

    anchor_ratios = globals().get(
        "COMPONENT_LABEL_RATIOS",
        [0.01, 0.09, 0.18, 0.30, 1.00],
    )

    for p in pair_table.itertuples(index=False):
        ratio = float(p.matched_ratio)
        label = f"{ratio:.2f}"

        if any(
            np.isclose(ratio, anchor, atol=1e-12)
            for anchor in anchor_ratios
        ):
            label += (
                f"\nS{int(p.spatial_K)}"
                f"/T{int(p.temporal_K)}"
            )

        labels.append(label)

    return labels


def plot_control_heatmaps(
    *,
    top_m_values,
    max_temporal_fraction,
    title,
    figure_note,
    png_path,
    pdf_path,
    figsize,
):
    arrays = {
        condition: make_control_arrays(
            condition,
            top_m_values,
            max_temporal_fraction=max_temporal_fraction,
        )
        for condition in CONDITIONS
    }

    fig, axes = plt.subplots(
        3,
        1,
        figsize=figsize,
        sharex=True,
        constrained_layout=False,
    )

    images = []

    for ax, condition in zip(axes, CONDITIONS):
        effect, robust = arrays[condition]

        im = ax.imshow(
            effect,
            aspect="auto",
            origin="lower",
            interpolation="nearest",
            cmap="RdBu_r",
            vmin=-max_abs,
            vmax=max_abs,
        )
        images.append(im)

        ax.set_title(
            CONDITION_TITLES[condition],
            fontsize=15,
            fontweight="bold",
        )
        ax.set_ylabel("top-m")
        ax.set_yticks(np.arange(len(top_m_values)))
        ax.set_yticklabels(top_m_values)

        for i in range(robust.shape[0]):
            for j in range(robust.shape[1]):
                if not robust[i, j]:
                    continue

                value = effect[i, j]

                ax.plot(
                    j,
                    i,
                    marker="o",
                    markersize=4.5,
                    markerfacecolor="none",
                    markeredgecolor=(
                        "white"
                        if (
                            np.isfinite(value)
                            and abs(value) > 0.40 * max_abs
                        )
                        else "black"
                    ),
                    markeredgewidth=1.1,
                    linestyle="None",
                )

    axes[-1].set_xticks(np.arange(len(pair_table)))
    axes[-1].set_xticklabels(
        component_ratio_labels(),
        rotation=45,
        ha="right",
        fontsize=9,
    )
    axes[-1].set_xlabel(
        "Matched component ratio (K / D)\n"
        "Selected labels show spatial K / temporal K"
    )

    fig.suptitle(
        title,
        fontsize=17,
        fontweight="bold",
        y=0.985,
    )

    fig.subplots_adjust(
        left=0.10,
        right=0.88,
        top=0.92,
        bottom=0.16,
        hspace=0.18,
    )

    color_ax = fig.add_axes([0.90, 0.20, 0.018, 0.60])
    colorbar = fig.colorbar(images[0], cax=color_ax)
    colorbar.set_label(
        "Spatial - temporal held-out accuracy"
    )

    fig.text(
        0.88,
        0.112,
        figure_note,
        ha="right",
        va="center",
        fontsize=9,
    )

    fig.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight",
    )
    fig.savefig(
        pdf_path,
        bbox_inches="tight",
    )
    plt.show()

    print("Saved:", png_path)
    print("Saved:", pdf_path)


plot_control_heatmaps(
    top_m_values=SPARSE_TOP_M_VALUES,
    max_temporal_fraction=None,
    title=(
        "Held-out spatial minus temporal decoding: "
        "sparse subsets"
    ),
    figure_note=(
        "Restricted to top-m <= 15; "
        "open circles use the notebook's robust-effect criterion"
    ),
    png_path=FIG_SPARSE_PNG,
    pdf_path=FIG_SPARSE_PDF,
    figsize=(13, 6.7),
)


plot_control_heatmaps(
    top_m_values=TOP_M_VALUES,
    max_temporal_fraction=MAX_TEMPORAL_SUBSET_FRACTION,
    title=(
        "Held-out spatial minus temporal decoding: "
        "temporal subset <= 50%"
    ),
    figure_note=(
        "Cells with m / temporal K > 0.50 are masked; "
        "this is a saturation diagnostic"
    ),
    png_path=FIG_MASKED_PNG,
    pdf_path=FIG_MASKED_PDF,
    figsize=(13, 9),
)

In [ ]:
requirement_rows = []

spatial_available_rows = set(
    zip(
        spatial["K"].astype(int),
        spatial["top_m"].astype(int),
    )
)
temporal_available_rows = set(
    zip(
        temporal["K"].astype(int),
        temporal["top_m"].astype(int),
    )
)

for p in pair_table.itertuples(index=False):
    spatial_K = int(p.spatial_K)
    temporal_K = int(p.temporal_K)

    if spatial_K % 7 != 0 or temporal_K % 3 != 0:
        continue

    n_spatial = spatial_K // 7
    n_temporal = temporal_K // 3

    if n_spatial != n_temporal:
        continue

    n = n_spatial

    for target_fraction in MATCHED_FRACTION_TARGETS:
        q = int(np.floor(target_fraction * n + 0.5))
        q = min(max(q, 1), n)

        spatial_m = 7 * q
        temporal_m = 3 * q
        actual_fraction = q / n

        requirement_rows.append({
            "spatial_K": spatial_K,
            "temporal_K": temporal_K,
            "matched_component_ratio": float(
                p.matched_ratio
            ),
            "target_retained_fraction": target_fraction,
            "actual_matched_fraction": actual_fraction,
            "spatial_top_m": spatial_m,
            "temporal_top_m": temporal_m,
            "spatial_row_available": (
                spatial_K,
                spatial_m,
            ) in spatial_available_rows,
            "temporal_row_available": (
                temporal_K,
                temporal_m,
            ) in temporal_available_rows,
        })

matched_fraction_requirements = pd.DataFrame(
    requirement_rows
)

matched_fraction_requirements[
    "both_rows_available"
] = (
    matched_fraction_requirements[
        "spatial_row_available"
    ]
    & matched_fraction_requirements[
        "temporal_row_available"
    ]
)

matched_fraction_requirements.to_csv(
    MATCHED_FRACTION_REQUIREMENTS_CSV,
    index=False,
)

display(matched_fraction_requirements)

print(
    "Exact matched-fraction configurations currently available:",
    int(
        matched_fraction_requirements[
            "both_rows_available"
        ].sum()
    ),
    "/",
    len(matched_fraction_requirements),
)
print(
    "Saved:",
    MATCHED_FRACTION_REQUIREMENTS_CSV,
)